### The goal of this project is to create a real-time invisibility cloak effect using computer vision.

The program:
1. Captures the background using the webcam.
2. Detects a blue-colored cloth using HSV color segmentation.
3. Creates a mask for the blue cloak.
4. Replaces the detected cloak region with the previously captured background.
5. Displays the final output in real time.
6. Optionally saves the output as a video file.

# Import Libraries

In [31]:
import cv2
import numpy as np
import time
from IPython.display import display, Image, clear_output

# Settings

In [32]:
# Set this to True to save the output video
SAVE_OUTPUT = False

# Name of the output file
OUTPUT_FILE = "blue_invisibility_cloak.mp4"

# Camera index (0 usually represents the default webcam)
CAMERA_INDEX = 0

# Open the Webcam

In [33]:
print("Starting Blue Invisibility Cloak Project...")
print("Please move out of the camera frame.")
print("The background will be captured in 3 seconds.")

# Open the webcam
cap = cv2.VideoCapture(CAMERA_INDEX)

# Check whether the camera opened successfully
if not cap.isOpened():
    print("Error: Could not open the camera.")
    raise SystemExit

# Give the user time to move out of the frame
time.sleep(3)

Starting Blue Invisibility Cloak Project...
Please move out of the camera frame.
The background will be captured in 3 seconds.


# Capture the Background

In [34]:
background = None

# Capture multiple frames
# The last successfully captured frame will be used as the background
for i in range(30):

    ret, frame = cap.read()

    if not ret:
        print("Error: Could not capture the background.")
        cap.release()
        raise SystemExit

    background = frame

# Mirror the background so it matches the mirrored live video
background = cv2.flip(background, 1)

print("Background captured successfully!")

Background captured successfully!


# Getting Frame Information & Output Video Setup

In [35]:
# Get the dimensions of the camera frame
height, width = background.shape[:2]

# Create the video writer only if saving is enabled
out = None

if SAVE_OUTPUT:

    # Define video codec
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    # Create VideoWriter object
    out = cv2.VideoWriter(
        OUTPUT_FILE,
        fourcc,
        20.0,
        (width, height)
    )

    print(f"Output will be saved as: {OUTPUT_FILE}")

Output will be saved as: blue_invisibility_cloak.mp4


# Main Invisibility Cloak Logic

In [ ]:
try:

    while cap.isOpened():

        # 1. CAPTURE A FRAME

        ret, img = cap.read()

        if not ret:
            print("Could not read camera frame.")
            break

        # Mirror the live video
        img = cv2.flip(img, 1)

        # 2. CONVERT BGR → HSV

        # HSV makes it easier to detect a specific color
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

        # 3. DETECT BLUE CLOAK

        # HSV range for blue
        lower_blue = np.array([100, 70, 25])
        upper_blue = np.array([130, 255, 200])

        # Create binary mask
        # White → detected blue pixels
        # Black → everything else
        mask = cv2.inRange(
            hsv,
            lower_blue,
            upper_blue
        )

        # 4. CLEAN THE MASK

        kernel = np.ones((3, 3), np.uint8)

        # Morphological opening removes small noise
        mask = cv2.morphologyEx(
            mask,
            cv2.MORPH_OPEN,
            kernel
        )

        # Dilation expands the detected cloak region and helps fill small gaps
        mask = cv2.morphologyEx(
            mask,
            cv2.MORPH_DILATE,
            kernel
        )

        # 5. CREATE INVERSE MASK

        # mask: White → blue cloak, Black → everything else
        # mask_inv: Black → blue cloak, White → everything else

        mask_inv = cv2.bitwise_not(mask)

        # 6. GET BACKGROUND WHERE CLOAK EXISTS

        # Replace the detected blue region with the previously captured background

        background_part = cv2.bitwise_and(
            background,
            background,
            mask=mask
        )

        # 7. KEEP LIVE VIDEO EVERYWHERE ELSE

        # Keep the live frame wherever the cloak was not detected

        current_part = cv2.bitwise_and(
            img,
            img,
            mask=mask_inv
        )

        # 8. COMBINE BOTH PARTS

        # Background where the cloak exists + Live video everywhere else

        final_output = cv2.add(
            background_part,
            current_part
        )

        # 9. DISPLAY LIVE OUTPUT IN JUPYTER

        success, encoded = cv2.imencode(
            ".jpg",
            final_output
        )

        if success:

            clear_output(wait=True)

            display(
                Image(data=encoded.tobytes())
            )

        # 10. OPTIONAL: SAVE OUTPUT

        # SAVE_OUTPUT = False → live display only
        # SAVE_OUTPUT = True → live display + MP4

        if SAVE_OUTPUT and out is not None:
            out.write(final_output)

except KeyboardInterrupt:

    print("\nStopping camera...")

finally:

    # Release webcam
    cap.release()

    # Finalize the video file if saving
    if out is not None:
        out.release()

    cv2.destroyAllWindows()

    print("Camera released successfully.")

    if SAVE_OUTPUT:
        print(f"Output saved as: {OUTPUT_FILE}")